# Annotation tool

This notebook uses [annotation_config.json](annotation_config.json) to configure the image folders and the ground-truth export CSV.

Run the cell below to start: click the corner points for each image (min/max configurable), use Previous/Next/Skip/Undo to navigate. Points are saved to the configured output CSV as you go.

**Trying it out for the first time?** The default config points at `data/uke/train`/`val`/`test`, which are empty until you put something there. The next cell runs [download_assets.py](../finetune/download_assets.py) to fetch the SmartDoc15 dataset (skipped if already downloaded), and the cell after that runs [create_dummy_uke_dataset.py](../finetune/create_dummy_uke_dataset.py) from [src/finetune](../finetune), which copies a sample of it into `data/uke/train`/`val`/`test` (relabeled as synthetic patients) with their existing ground truth already filled in — so you'll see pre-drawn corners here that you can inspect, tweak, and re-save, rather than starting from a blank image. Skip or delete both cells once you're pointing the config at your own images.

**Using your own data?** Edit [annotation_config.json](annotation_config.json):
- `image_folders`: one or more folders containing the images you want to annotate
- `output_csv`: where the annotated polygon points get written
- `min_points`/`max_points`: allowed polygon corner count (default `4`/`50`)

In [ ]:
# Optional, one-time: download the pretrained U2NET weights and the SmartDoc15 dataset.
# The "dummy UKE dataset" cell below needs this, since it copies its ground-truth CSV
# from the downloaded SmartDoc15 data. Skips downloads that are already present, so it's
# safe to re-run.
import subprocess
import sys
from pathlib import Path

_finetune_dir = Path.cwd().parent / "finetune"
subprocess.run([sys.executable, "download_assets.py"], cwd=_finetune_dir, check=False)

In [ ]:
# Optional, one-time: copy a SmartDoc15-based example dataset (with ground truth already
# filled in) into data/uke/train|val|test. Requires SmartDoc15 to already be downloaded
# (see the cell above).
# Skip or delete this cell once you're annotating your own images. Safe to re-run.
import subprocess
import sys
from pathlib import Path

_finetune_dir = Path.cwd().parent / "finetune"
subprocess.run([sys.executable, "create_dummy_uke_dataset.py"], cwd=_finetune_dir, check=False)

In [ ]:
%matplotlib widget

import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

import annotation as annotation_module
importlib.reload(annotation_module)

from annotation import (
    collect_image_files,
    init_context,
    load_annotation_config,
    load_annotation_dataframe,
    onclick,
    on_next,
    on_prev,
    on_skip,
    on_undo,
    show_image,
 )

# --- Configuration ---
config = load_annotation_config()
config_path = config["config_path"]
image_folders = config["image_folders"]
output_csv = config["output_csv"]
min_points = config["min_points"]
max_points = config["max_points"]

print(f"Annotation config: {config_path}")
print("Image folders:")
for folder in image_folders:
    print(f" - {folder}")
print(f"GT output CSV: {output_csv}")

base_columns = [
    "image_path",
    "model_width",
    "model_height",
    "polygon_points",
    "tl_x", "tl_y",
    "bl_x", "bl_y",
    "br_x", "br_y",
    "tr_x", "tr_y",
]

df = load_annotation_dataframe(output_csv, base_columns)
image_files = collect_image_files(image_folders, config["image_extensions"] )
current_index = 0

# --- Plot setup ---
fig, ax = plt.subplots()
fig.canvas.header_visible = False
scatter_points = []
points = []

cid = fig.canvas.mpl_connect("button_press_event", onclick)

prev_button = widgets.Button(description="Previous image")
prev_button.on_click(on_prev)
next_button = widgets.Button(description="Next image")
next_button.on_click(on_next)
skip_button = widgets.Button(description="Skip")
skip_button.on_click(on_skip)
undo_button = widgets.Button(description="Delete last point")
undo_button.on_click(on_undo)
display(widgets.HBox([prev_button, next_button, skip_button, undo_button]))

init_context(
    df=df,
    image_files=image_files,
    current_index=current_index,
    min_points=min_points,
    max_points=max_points,
    output_csv=output_csv,
    fig=fig,
    ax=ax,
    points=points,
    scatter_points=scatter_points,
    prev_button=prev_button,
    next_button=next_button,
    skip_button=skip_button,
    undo_button=undo_button,
    cid=cid,
    base_columns=base_columns,
    image_folders=image_folders,
 )

if len(image_files) == 0:
    print("No images found in the configured folders.")
    prev_button.disabled = True
    next_button.disabled = True
    undo_button.disabled = True
    skip_button.disabled = True
else:
    model_width, model_height = show_image(current_index)
    img_path = image_files[current_index]
    rows = df[df.image_path.astype(str).str.endswith(img_path.name)] if "image_path" in df.columns else pd.DataFrame()
    has_polygon = False
    if len(rows):
        exact = rows[rows.image_path.astype(str) == str(img_path)]
        if len(exact):
            r = exact.iloc[0]
        else:
            r = rows.iloc[0]
        if pd.notna(r.polygon_points) and str(r.polygon_points).strip() != "":
            has_polygon = True
    if not has_polygon:
        points.clear()
        points.extend([(0, 0), (model_width, 0), (model_width, model_height), (0, model_height)])
        for s in list(scatter_points):
            try:
                s.remove()
            except Exception:
                pass
        scatter_points.clear()
        p = ax.scatter([pt[0] for pt in points], [pt[1] for pt in points], c="lime", s=40)
        scatter_points.append(p)
        fig.canvas.draw_idle()